# ISTAT housing crowding: densita' abitativa media per titolo di godimento

Supporto analitico minimo per una prima discussion `Analisi`.

- **Obiettivo**: confrontare la densita' media di componenti per 100 mq tra abitazioni in affitto e di proprieta'.
- **Dati**: mart `mart_crowding_tenure_italy` (ISTAT dataflow 33_179).
- **Orizzonte usato qui**: serie annuale 2004-2024.
- **Nota**: questa metrica non coincide con il tasso standard di sovraffollamento europeo.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

parquet_path = Path("../../../dataset-incubator/out/data/mart/istat_housing_crowding/2024/mart_crowding_tenure_italy.parquet")

if not parquet_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {parquet_path.resolve()}")

con = duckdb.connect()
parquet_path

In [ ]:
query = """
SELECT
    anno,
    titolo_godimento,
    componenti_per_100mq
FROM read_parquet(?)
ORDER BY anno ASC, titolo_godimento DESC
"""

df_res = con.execute(query, [str(parquet_path)]).df()
df_pivot = df_res.pivot(index='anno', columns='titolo_godimento', values='componenti_per_100mq')
df_pivot['diff_rent_prop'] = df_pivot['rent'] - df_pivot['property']
df_pivot.tail(5)

In [ ]:
spotlight_years = [2004, 2024]
spotlight = df_pivot.loc[df_pivot.index.isin(spotlight_years), ['property', 'rent', 'diff_rent_prop']]
spotlight

### Caveat

- **Dato campionario**: fonte ISTAT EU-SILC.
- **Densita' vs sovraffollamento**: componenti per 100 mq, non stanze per persona.
- **Composizione familiare**: qui non distinguiamo ancora per tipo di nucleo o presenza di figli.
- **Perimetro**: questa e' una prima vista nazionale e descrittiva, non una misura completa di disagio abitativo.